# Pandas Performance Optimization 

In [1]:
import pandas as pd
import numpy as np
import time

np.random.seed(42)

# Create a large DataFrame for testing (1,000,000 rows)
n_rows = 1_000_000
df = pd.DataFrame({
    'A': np.random.randint(1, 100, n_rows),
    'B': np.random.randint(1, 100, n_rows),
    'C': np.random.randn(n_rows),
    'D': np.random.choice(['X', 'Y', 'Z'], n_rows),
    'E': np.random.randint(1, 1000, n_rows)
})
print(f"DataFrame shape: {df.shape}")
print(df.head())

DataFrame shape: (1000000, 5)
    A   B         C  D    E
0  52  83 -0.433531  Y  682
1  93  52  1.730537  Z  424
2  15  85  0.383007  Y  646
3  72  28  0.059450  X  907
4  61  92 -0.767006  Z  208


# Pandas vertorization vs pyhton loops 

In [2]:

# SLOW WAY: Using a for loop

print("\n--- 1. Vectorization vs. Loops ---")

start_time = time.time()
# Create a copy to avoid modifying the original
df_loop = df.copy()

# Creating a new column for existing columns A, B, and E for testing 

for i in range(len(df_loop)):
    df_loop.loc[i, 'new_col_loop'] = df_loop.loc[i, 'A'] * df_loop.loc[i, 'B'] + df_loop.loc[i, 'E']
loop_duration = time.time() - start_time
print(f"For loop time: {loop_duration:.4f} seconds")


# FAST WAY: Vectorized operation

start_time = time.time()
df_vec = df.copy()
df_vec['new_col_vec'] = df_vec['A'] * df_vec['B'] + df_vec['E']
vectorized_duration = time.time() - start_time
print(f"Vectorized time: {vectorized_duration:.4f} seconds")
print(f"Vectorization is {loop_duration / vectorized_duration:.1f}x faster!")

# Verify results are the same
print(f"Results match: {(df_loop['new_col_loop'] == df_vec['new_col_vec']).all()}")


--- 1. Vectorization vs. Loops ---
For loop time: 235.0109 seconds
Vectorized time: 0.0123 seconds
Vectorization is 19068.9x faster!
Results match: True


# Pandas Category Datatype for string 

In [3]:

print("Memory usage before optimization:")
print(df.info(memory_usage='deep'))

# Calculate memory of the object column specifically
mem_before = df['D'].memory_usage(deep=True)

# converting into category data type
df['D_category'] = df['D'].astype('category')
mem_after = df['D_category'].memory_usage(deep=True)

print(f"\nMemory for column 'D' (object): {mem_before / 1024:.2f} KB")
print(f"Memory for column 'D_category' (category): {mem_after / 1024:.2f} KB")
print(f"Memory reduction: {(1 - mem_after / mem_before) * 100:.1f}%")

Memory usage before optimization:
<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 5 columns):
 #   Column  Non-Null Count    Dtype  
---  ------  --------------    -----  
 0   A       1000000 non-null  int32  
 1   B       1000000 non-null  int32  
 2   C       1000000 non-null  float64
 3   D       1000000 non-null  str    
 4   E       1000000 non-null  int32  
dtypes: float64(1), int32(3), str(1)
memory usage: 66.8 MB
None

Memory for column 'D' (object): 48828.25 KB
Memory for column 'D_category' (category): 976.84 KB
Memory reduction: 98.0%


# Chunking original dataframe if dataframe is loo large 

In [4]:

df_mem = df[['A', 'B', 'E']].copy()
print("Original memory for int64 columns:")
print(df_mem.info(memory_usage='deep'))

# integer downcasting
for col in ['A', 'B', 'E']:
    df_mem[col] = pd.to_numeric(df_mem[col], downcast='integer')

print(df_mem.info(memory_usage='deep'))


dummy_file = 'large_dummy.csv'
df[['A', 'B', 'C']].to_csv(dummy_file, index=False)

chunk_size = 100000
total_sum = 0
chunk_count = 0

start_time = time.time()
for chunk in pd.read_csv(dummy_file, chunksize=chunk_size):
    # Perform an operation on each chunk (e.g., sum of column 'A')
    total_sum += chunk['A'].sum()
    chunk_count += 1
    print(f"  Processed chunk {chunk_count}")

chunk_duration = time.time() - start_time
print(f"Total sum from chunks: {total_sum}")
print(f"Chunk processing time: {chunk_duration:.4f} seconds")

# Clean up the dummy file
import os
os.remove(dummy_file)

Original memory for int64 columns:
<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 3 columns):
 #   Column  Non-Null Count    Dtype
---  ------  --------------    -----
 0   A       1000000 non-null  int32
 1   B       1000000 non-null  int32
 2   E       1000000 non-null  int32
dtypes: int32(3)
memory usage: 11.4 MB
None
<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 3 columns):
 #   Column  Non-Null Count    Dtype
---  ------  --------------    -----
 0   A       1000000 non-null  int8 
 1   B       1000000 non-null  int8 
 2   E       1000000 non-null  int16
dtypes: int16(1), int8(2)
memory usage: 3.8 MB
None
  Processed chunk 1
  Processed chunk 2
  Processed chunk 3
  Processed chunk 4
  Processed chunk 5
  Processed chunk 6
  Processed chunk 7
  Processed chunk 8
  Processed chunk 9
  Processed chunk 10
Total sum from chunks: 49966419
Chunk processing time: 0.3349 seconds


# using indexing for fast lookups

In [5]:

# Create a DataFrame with a unique ID column
df_index = pd.DataFrame({
    'user_id': range(100000, 0, -1),
    'data': np.random.randn(100000)
})

# Search without an index
start_time = time.time()
result_no_index = df_index[df_index['user_id'] == 50000]
no_index_duration = time.time() - start_time
print(f"Search on column (no index) time: {no_index_duration:.4f} seconds")

# Set the index
df_index.set_index('user_id', inplace=True)

# Search with the index
start_time = time.time()
result_with_index = df_index.loc[50000]
with_index_duration = time.time() - start_time
print(f"Search with index time: {with_index_duration:.4f} seconds")

if with_index_duration > 0:
    print(f"Indexed lookup is {no_index_duration / with_index_duration:.1f}x faster!")
else:
    print("Indexed lookup was effectively instantaneous.")

Search on column (no index) time: 0.0023 seconds
Search with index time: 0.0001 seconds
Indexed lookup is 18.5x faster!
